# 03 AI Capacity and Capex Options

RPO is not the same as equity value. This notebook keeps upside from
additional AI capacity separate from downside tied to capex, financing,
customer concentration and delayed conversion.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from valuation import (
    DcfInputs,
    MarketPremiumInputs,
    OptionScenario,
    SegmentValuation,
    SotpInputs,
    build_sensitivity_grid,
    discounted_cash_flow,
    implied_margin_for_enterprise_value,
    implied_revenue_for_enterprise_value,
    market_premium_value,
    probability_weighted_option_value,
    sotp_equity_value,
)

DATA_DIR = next(
    candidate / "apps/notebooks/studies/oracle_valuation/data"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "apps/notebooks/studies/oracle_valuation/data").exists()
)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
options = pd.read_csv(DATA_DIR / "option_scenarios.csv")
options

In [ ]:
option_values = {}
for option_name, group in options.groupby("option"):
    scenarios = [
        OptionScenario(row.case, row.probability, row.value_usd_bn)
        for row in group.itertuples(index=False)
    ]
    option_values[option_name] = probability_weighted_option_value(scenarios)
option_values

In [ ]:
capex_funding_grid = build_sensitivity_grid(
    row_values=[60, 70, 80, 90, 100],
    column_values=[0.00, 0.15, 0.25, 0.35],
    formula=lambda gross_capex, customer_funded_pct: gross_capex * (1 - customer_funded_pct),
    row_name="FY2027 gross capex (USD bn)",
    column_name="Customer-funded percentage",
)
capex_funding_grid

In [ ]:
dilution_grid = build_sensitivity_grid(
    row_values=[175, 185, 200, 225],
    column_values=[10, 20, 30, 40],
    formula=lambda share_price, equity_raise: equity_raise / share_price,
    row_name="Equity raise price ($)",
    column_name="Equity raise (USD bn)",
)
dilution_grid

In [ ]:
option_frame = pd.DataFrame(
    [{"option": name, "probability_weighted_value_usd_bn": value} for name, value in option_values.items()]
)
ax = option_frame.plot.barh(
    x="option",
    y="probability_weighted_value_usd_bn",
    figsize=(8, 3),
    title="Probability-Weighted Oracle AI/Capex Options",
)
ax.set_xlabel("USD bn")
plt.tight_layout()
option_frame